<a href="https://colab.research.google.com/github/appconta948-sys/tuamigocontable.com/blob/main/contaapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# ============================================
# CELDA 1: Montar Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================
# CELDA 2: Instalar dependencias
# ============================================
!pip install -q google-generativeai pandas matplotlib ipywidgets openpyxl xlsxwriter requests

# ============================================
# CELDA 3: Importaciones
# ============================================
import sys
import json
import os
import hashlib
import uuid
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import requests
from google.colab import userdata
import google.generativeai as genai
print("✅ Librerías importadas")

# ============================================
# CELDA 4: Clase Conta
# ============================================
class Conta:
    def __init__(self, pais="Colombia", moneda="COP", grupoNIIF=2):
        self.pais = pais
        self.moneda = moneda
        self.grupoNIIF = grupoNIIF
        self.experiencia = "20 años"
        self.objetivos = {
            "general": "Pensar como contador con 20 años de experiencia",
            "especificos": [
                "Definir qué es contabilidad",
                "Conocer las áreas de la contabilidad",
                "Identificar objetivos de contabilidad financiera y administrativa",
                "Conocer ley que regula la profesión contable",
                "Identificar finalidad real de la contabilidad",
                "Conocer principios básicos y NIIF"
            ]
        }
        self.areas = {
            "financiera": {
                "activa": True,
                "proposito": "Recopilar, clasificar y registrar eventos económicos",
                "productos": ["Balance General", "Estado de Resultados", "Estado de Patrimonio", "Flujo de Efectivo"]
            },
            "administrativa": {
                "activa": True,
                "proposito": "Interpretar información para toma de decisiones",
                "indices": ["Rotación Inventario", "Cuentas por Cobrar", "Apalancamiento", "Razón Corriente", "Prueba Ácido"]
            },
            "impositiva": {
                "activa": True,
                "proposito": "Manejar aspectos fiscales",
                "impuestos": ["Renta", "IVA/ITBMS", "Licencia comercial", "Dividendos"]
            }
        }
        self.finalidad = {
            "revision": {"descripcion": "Examinar información antes de registrar", "documentos": ["facturas", "recibos", "cheques"]},
            "clasificacion": {"descripcion": "Asignar código PUC", "regla": "Identificar cuentas débito/crédito"},
            "anotacion": {"descripcion": "Registrar en libros oficiales", "tipo": "asientos contables"},
            "informacion": {"descripcion": "Presentar en estados financieros", "productos": ["Balance", "PyG", "Patrimonio", "Flujo"]},
            "interpretacion": {"descripcion": "Analizar con índices", "objetivo": "Toma de decisiones"}
        }
        self.principios = {
            "enteEconomico": "La empresa es distinta de sus dueños",
            "continuidad": "Se asume que la empresa seguirá operando",
            "unidadMedida": "Moneda local",
            "valuacion": "Costo histórico",
            "esenciaSobreForma": "Realidad económica sobre forma legal",
            "realizacion": "Registro cuando ocurre, no cuando se paga/cobra",
            "asociacion": "Ingresos y gastos en mismo período",
            "revelacionSuficiente": "Información completa para decisiones"
        }
    def pensar(self, transaccion):
        return {"analisis": "Transacción analizada", "clasificacion": self.clasificarTransaccion(transaccion)}
    def clasificarTransaccion(self, transaccion):
        if transaccion.get("tipo") in ["venta", "compra"]:
            return {"area": "financiera", "subarea": "operativa"}
        elif transaccion.get("impuestos"):
            return {"area": "impositiva", "subarea": "fiscal"}
        return {"area": "general"}
print("✅ Clase Conta definida")

# ============================================
# CELDA 5: Clase PUCInteligente
# ============================================
class PUCInteligente:
    def __init__(self, conta, pais="Colombia"):
        self.conta = conta
        self.pais = pais
        self.cuentas = {}
        self.inicializarPUC()
    def inicializarPUC(self):
        # CLASE 1: ACTIVO
        self.cuentas["1"] = {"codigo": "1", "nombre": "ACTIVO", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 1, "aceptaMovimientos": False}
        self.cuentas["11"] = {"codigo": "11", "nombre": "ACTIVO CORRIENTE", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 2, "padre": "1", "aceptaMovimientos": False}
        self.cuentas["1105"] = {"codigo": "1105", "nombre": "EFECTIVO", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 3, "padre": "11", "aceptaMovimientos": True, "requiereTercero": False, "requiereDocumentoSoporte": True}
        self.cuentas["110505"] = {"codigo": "110505", "nombre": "CAJA", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 4, "padre": "1105", "aceptaMovimientos": True, "requiereTercero": False, "requiereDocumentoSoporte": True}
        self.cuentas["110510"] = {"codigo": "110510", "nombre": "BANCOS", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 4, "padre": "1105", "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
        self.cuentas["1305"] = {"codigo": "1305", "nombre": "CLIENTES", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 4, "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
        self.cuentas["1405"] = {"codigo": "1405", "nombre": "INVENTARIO", "naturaleza": "DEBITO", "clase": "BALANCE", "nivel": 4, "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
        # CLASE 2: PASIVO
        self.cuentas["2"] = {"codigo": "2", "nombre": "PASIVO", "naturaleza": "CREDITO", "clase": "BALANCE", "nivel": 1, "aceptaMovimientos": False}
        self.cuentas["2105"] = {"codigo": "2105", "nombre": "PROVEEDORES", "naturaleza": "CREDITO", "clase": "BALANCE", "nivel": 3, "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
        self.cuentas["2408"] = {"codigo": "2408", "nombre": "IMPUESTOS POR PAGAR", "naturaleza": "CREDITO", "clase": "BALANCE", "nivel": 3, "aceptaMovimientos": True, "requiereDocumentoSoporte": True}
        # CLASE 4: INGRESOS
        self.cuentas["4"] = {"codigo": "4", "nombre": "INGRESOS", "naturaleza": "CREDITO", "clase": "ESTADO_RESULTADOS", "nivel": 1, "aceptaMovimientos": False}
        self.cuentas["4135"] = {"codigo": "4135", "nombre": "VENTAS", "naturaleza": "CREDITO", "clase": "ESTADO_RESULTADOS", "nivel": 3, "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
        # CLASE 5: GASTOS
        self.cuentas["5"] = {"codigo": "5", "nombre": "GASTOS", "naturaleza": "DEBITO", "clase": "ESTADO_RESULTADOS", "nivel": 1, "aceptaMovimientos": False}
        self.cuentas["5105"] = {"codigo": "5105", "nombre": "GASTOS PERSONAL", "naturaleza": "DEBITO", "clase": "ESTADO_RESULTADOS", "nivel": 3, "aceptaMovimientos": True, "requiereTercero": True, "requiereDocumentoSoporte": True}
    def obtenerCuenta(self, codigo):
        return self.cuentas.get(codigo)
    def validarMovimiento(self, codigoCuenta, movimiento):
        cuenta = self.obtenerCuenta(codigoCuenta)
        if not cuenta:
            return {"valido": False, "error": "Cuenta no existe"}
        if not cuenta.get("aceptaMovimientos", False):
            return {"valido": False, "error": "Cuenta no acepta movimientos directos"}
        return {"valido": True, "cuenta": cuenta}
print("✅ Clase PUC definida")

# ============================================
# CELDA 6: Clase LibroDiario
# ============================================
class LibroDiario:
    def __init__(self, conta, puc):
        self.conta = conta
        self.puc = puc
        self.asientos = []
        self.secuencial = 1
    def registrarAsiento(self, transaccion):
        try:
            for mov in transaccion["movimientos"]:
                validacion = self.puc.validarMovimiento(mov["cuenta"], {})
                if not validacion["valido"]:
                    raise Exception(f"Cuenta {mov['cuenta']} inválida")
            asiento = {
                "id": f"A-{self.secuencial:04d}",
                "fecha": transaccion.get("fecha", datetime.now().strftime("%Y-%m-%d")),
                "comprobante": transaccion.get("comprobante", f"CJ-{self.secuencial:04d}"),
                "descripcion": transaccion.get("descripcion", ""),
                "tercero": transaccion.get("tercero", ""),
                "documento": transaccion.get("documentoSoporte", ""),
                "movimientos": [],
                "totalDebito": 0,
                "totalCredito": 0
            }
            for mov in transaccion["movimientos"]:
                cuenta = self.puc.obtenerCuenta(mov["cuenta"])
                movimiento = {
                    "cuenta": mov["cuenta"],
                    "nombre": cuenta["nombre"] if cuenta else "Desconocida",
                    "detalle": mov.get("detalle", ""),
                    "debito": mov["valor"] if mov["tipo"] == "DEBITO" else 0,
                    "credito": mov["valor"] if mov["tipo"] == "CREDITO" else 0
                }
                asiento["movimientos"].append(movimiento)
                asiento["totalDebito"] += movimiento["debito"]
                asiento["totalCredito"] += movimiento["credito"]
            if abs(asiento["totalDebito"] - asiento["totalCredito"]) > 0.01:
                raise Exception("No cuadra partida doble")
            self.asientos.append(asiento)
            self.secuencial += 1
            return {"exito": True, "asiento": asiento}
        except Exception as e:
            return {"exito": False, "error": str(e)}
    def obtener_ingresos_mes(self):
        total = 0
        for a in self.asientos:
            for m in a["movimientos"]:
                if m["cuenta"] in ["4135"] and m["credito"] > 0:
                    total += m["credito"]
        return total
    def obtener_egresos_mes(self):
        total = 0
        for a in self.asientos:
            for m in a["movimientos"]:
                if m["cuenta"] in ["5105", "5115"] and m["debito"] > 0:
                    total += m["debito"]
        return total
    def obtener_balance(self):
        return self.obtener_ingresos_mes() - self.obtener_egresos_mes()
print("✅ Clase LibroDiario definida")

# ============================================
# CELDA 7: Clase ContaInteligente (IA)
# ============================================
class ContaInteligente:
    def __init__(self, conta=None, puc=None, libro=None):
        self.conta = conta
        self.puc = puc
        self.libro = libro
        self.disponible = False
        try:
            self.api_key = userdata.get('GEMINI_API_KEY')
            if self.api_key:
                genai.configure(api_key=self.api_key)
                self.model = genai.GenerativeModel('gemini-2.0-flash')
                self.disponible = True
                print("✅ Gemini API conectado")
            else:
                print("⚠️ API key no encontrada")
        except Exception as e:
            print(f"⚠️ Error IA: {e}")
    def interpretar_mensaje(self, mensaje):
        if not self.disponible:
            return {"error": "IA no disponible"}
        prompt = f"""Eres un asistente contable. Interpreta este mensaje: "{mensaje}".
        Devuelve JSON con tipo, monto, tercero, producto, cantidad."""
        try:
            response = self.model.generate_content(prompt)
            return {"exito": True, "respuesta": response.text}
        except Exception as e:
            return {"error": str(e)}
print("✅ Clase ContaInteligente definida")

# ============================================
# CELDA 8: Inicializar sistema y datos de ejemplo
# ============================================
conta = Conta("Colombia", "COP", 2)
puc = PUCInteligente(conta, "Colombia")
libro = LibroDiario(conta, puc)
ia = ContaInteligente(conta, puc, libro)

ejemplos = [
    {
        "fecha": "2024-03-15",
        "comprobante": "FV-001",
        "descripcion": "Venta de mercancía",
        "tercero": "Cliente A",
        "documentoSoporte": "FAC-001",
        "movimientos": [
            {"cuenta": "1305", "tipo": "DEBITO", "valor": 1190000, "detalle": "Cliente"},
            {"cuenta": "4135", "tipo": "CREDITO", "valor": 1000000, "detalle": "Ventas"},
            {"cuenta": "2408", "tipo": "CREDITO", "valor": 190000, "detalle": "IVA"}
        ]
    },
    {
        "fecha": "2024-03-20",
        "comprobante": "FV-002",
        "descripcion": "Venta servicios",
        "tercero": "Cliente B",
        "documentoSoporte": "FAC-002",
        "movimientos": [
            {"cuenta": "110505", "tipo": "DEBITO", "valor": 500000, "detalle": "Efectivo"},
            {"cuenta": "4135", "tipo": "CREDITO", "valor": 500000, "detalle": "Ventas"}
        ]
    }
]

for ej in ejemplos:
    libro.registrarAsiento(ej)

print(f"\n✅ SISTEMA INICIALIZADO")
print(f"   Asientos registrados: {len(libro.asientos)}")
print(f"   IA disponible: {ia.disponible}")

# ============================================
# CELDA 9: Dashboard simple
# ============================================
def mostrar_dashboard():
    ingresos = libro.obtener_ingresos_mes()
    egresos = libro.obtener_egresos_mes()
    balance = ingresos - egresos
    html = f"""
    <div style="font-family: Arial; max-width: 1000px; margin: auto;">
        <h1>📊 CONTA APP - Dashboard</h1>
        <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 20px;">
            <div style="background: #4caf50; color: white; padding: 20px; border-radius: 10px;">
                <h3>Ingresos</h3>
                <p style="font-size: 24px;">${ingresos:,.0f}</p>
            </div>
            <div style="background: #f44336; color: white; padding: 20px; border-radius: 10px;">
                <h3>Egresos</h3>
                <p style="font-size: 24px;">${egresos:,.0f}</p>
            </div>
            <div style="background: #2196f3; color: white; padding: 20px; border-radius: 10px;">
                <h3>Balance</h3>
                <p style="font-size: 24px;">${balance:,.0f}</p>
            </div>
        </div>
        <h3>Últimos movimientos:</h3>
        <table style="width:100%; border-collapse: collapse;">
            <tr><th>Fecha</th><th>Descripción</th><th>Valor</th></tr>
    """
    for a in libro.asientos[-5:]:
        for m in a["movimientos"]:
            valor = m["debito"] or m["credito"]
            html += f"<tr><td>{a['fecha']}</td><td>{m['detalle'][:30]}</td><td>${valor:,.0f}</td></tr>"
    html += "</table></div>"
    display(HTML(html))

mostrar_dashboard()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Librerías importadas
✅ Clase Conta definida
✅ Clase PUC definida
✅ Clase LibroDiario definida
✅ Clase ContaInteligente definida
✅ Gemini API conectado

✅ SISTEMA INICIALIZADO
   Asientos registrados: 2
   IA disponible: True


Fecha,Descripción,Valor
2024-03-15,Cliente,"$1,190,000"
2024-03-15,Ventas,"$1,000,000"
2024-03-15,IVA,"$190,000"
2024-03-20,Efectivo,"$500,000"
2024-03-20,Ventas,"$500,000"
